In [ ]:
# %% [markdown]
# # Phân tích US Baby Names (1880-2010)
# 
# ## Mục tiêu
# - Khám phá xu hướng đặt tên trẻ em ở Mỹ trong hơn 130 năm
# - Tìm hiểu sự phổ biến của từng tên theo thời gian
# - Đánh giá sự đa dạng văn hóa và thay đổi giới tính trong đặt tên
# - Xác định các mô hình dài hạn về độ phổ biến và tác động xã hội

# %%
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("Thư viện đã được import thành công!")

# %% [markdown]
# ## 1. Load và Chuẩn bị Dữ liệu

# %%
# Load dữ liệu từ các file theo năm
# Lưu ý: Thay đổi đường dẫn phù hợp với môi trường của bạn

pieces = []
for year in range(1880, 2011):
    path = f"datasets/babynames/yob{year}.txt"
    frame = pd.read_csv(path, names=["name", "sex", "births"])
    frame["year"] = year
    pieces.append(frame)

# Concatenate tất cả dữ liệu
names = pd.concat(pieces, ignore_index=True)

print(f"Tổng số records: {len(names):,}")
print(f"Khoảng thời gian: {names['year'].min()} - {names['year'].max()}")
print(f"\nThông tin dữ liệu:")
names.info()

# %%
# Xem một số mẫu dữ liệu
print("5 records đầu tiên:")
print(names.head())
print("\n5 records cuối cùng:")
print(names.tail())

# %%
# Kiểm tra giá trị thiếu và trùng lặp
print(f"Số lượng giá trị thiếu:")
print(names.isnull().sum())
print(f"\nSố lượng records trùng lặp: {names.duplicated().sum()}")

# Thống kê cơ bản
print(f"\nThống kê mô tả:")
print(names.describe())

# %% [markdown]
# ## 2. PHÂN TÍCH MÔ TẢ

# %% [markdown]
# ### 2.1. Xu hướng Số lượng Sinh theo Giới tính (1880-2010)

# %%
# Tính tổng số sinh theo năm và giới tính
total_births = names.pivot_table("births", index="year", columns="sex", aggfunc=sum)

print("Tổng số sinh 5 năm cuối:")
print(total_births.tail())

# Vẽ biểu đồ xu hướng
fig, ax = plt.subplots(figsize=(14, 6))
total_births.plot(ax=ax, linewidth=2)
ax.set_title("Tổng số Trẻ Sinh theo Giới tính (1880-2010)", fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Số lượng Sinh", fontsize=12)
ax.legend(["Nữ", "Nam"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Phân tích tăng trưởng
print("\n=== PHÂN TÍCH TĂNG TRƯỞNG ===")
print(f"Năm 1880: Nam = {total_births.loc[1880, 'M']:,.0f}, Nữ = {total_births.loc[1880, 'F']:,.0f}")
print(f"Năm 2010: Nam = {total_births.loc[2010, 'M']:,.0f}, Nữ = {total_births.loc[2010, 'F']:,.0f}")
print(f"\nTăng trưởng Nam: {(total_births.loc[2010, 'M'] / total_births.loc[1880, 'M'] - 1) * 100:.1f}%")
print(f"Tăng trưởng Nữ: {(total_births.loc[2010, 'F'] / total_births.loc[1880, 'F'] - 1) * 100:.1f}%")

# %% [markdown]
# ### 2.2. Top 10 Tên Phổ biến theo Thập kỷ

# %%
# Tạo cột thập kỷ
names['decade'] = (names['year'] // 10) * 10

# Hàm tìm top N tên theo thập kỷ
def get_top_names_by_decade(df, n=10):
    result = {}
    for sex in ['M', 'F']:
        sex_data = df[df['sex'] == sex]
        top_names = sex_data.groupby(['decade', 'name'])['births'].sum().reset_index()
        top_names = top_names.sort_values(['decade', 'births'], ascending=[True, False])
        top_names = top_names.groupby('decade').head(n)
        result[sex] = top_names
    return result

top_names_decade = get_top_names_by_decade(names, n=10)

# Hiển thị top 10 tên của thập kỷ 2000
print("=== TOP 10 TÊN PHỔ BIẾN THẬP KỶ 2000 ===")
print("\nBé Trai:")
print(top_names_decade['M'][top_names_decade['M']['decade'] == 2000][['name', 'births']])
print("\nBé Gái:")
print(top_names_decade['F'][top_names_decade['F']['decade'] == 2000][['name', 'births']])

# %%
# Visualize top 5 tên mỗi giới qua các thập kỷ
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, sex in enumerate(['M', 'F']):
    sex_data = top_names_decade[sex]
    pivot_data = sex_data.pivot_table('births', index='decade', columns='name', aggfunc='sum')
    
    # Lấy top 5 tên có tổng births cao nhất
    top5_names = pivot_data.sum().nlargest(5).index
    pivot_data[top5_names].plot(ax=axes[idx], marker='o', linewidth=2)
    
    title = "Bé Trai" if sex == 'M' else "Bé Gái"
    axes[idx].set_title(f"Top 5 Tên Phổ biến - {title}", fontsize=14, fontweight='bold')
    axes[idx].set_xlabel("Thập kỷ", fontsize=11)
    axes[idx].set_ylabel("Tổng số Sinh", fontsize=11)
    axes[idx].legend(title="Tên", fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# %% [markdown]
# ### 2.3. Độ Tập trung Tên (Name Concentration)

# %%
# Tính tỷ lệ tên trong top 1000 so với tổng số sinh
def add_prop(group):
    group["prop"] = group["births"] / group["births"].sum()
    return group

names = names.groupby(["year", "sex"], group_keys=False).apply(add_prop)

# Tạo top 1000 dataset
def get_top1000(group):
    return group.sort_values("births", ascending=False)[:1000]

grouped = names.groupby(["year", "sex"])
top1000 = grouped.apply(get_top1000).reset_index(drop=True)

print(f"Số records trong top1000: {len(top1000):,}")
print("\nMẫu dữ liệu top1000:")
print(top1000.head())

# %%
# Tính tỷ lệ tích lũy của top 1000
table = top1000.pivot_table("prop", index="year", columns="sex", aggfunc=sum)

fig, ax = plt.subplots(figsize=(14, 6))
table.plot(ax=ax, linewidth=2)
ax.set_title("Tỷ lệ Sinh trong Top 1000 Tên (1880-2010)", fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Tỷ lệ Tích lũy", fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_yticks(np.linspace(0, 1, 11))
ax.legend(["Nữ", "Nam"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH ĐỘ TẬP TRUNG ===")
print(f"Năm 1880: Nam = {table.loc[1880, 'M']:.1%}, Nữ = {table.loc[1880, 'F']:.1%}")
print(f"Năm 2010: Nam = {table.loc[2010, 'M']:.1%}, Nữ = {table.loc[2010, 'F']:.1%}")
print("\n→ Độ tập trung giảm = tăng đa dạng trong đặt tên")

# %% [markdown]
# ## 3. PHÂN TÍCH KHÁM PHÁ

# %% [markdown]
# ### 3.1. Đo lường Độ Đa dạng (Name Diversity)

# %%
# Tính số lượng tên cần thiết để đạt 50% tổng số sinh
def get_quantile_count(group, q=0.5):
    group = group.sort_values("prop", ascending=False)
    return group.prop.cumsum().searchsorted(q) + 1

diversity = top1000.groupby(["year", "sex"]).apply(get_quantile_count)
diversity = diversity.unstack()

print("Số lượng tên chiếm 50% dân số:")
print(diversity.head(10))
print("\n...")
print(diversity.tail(10))

# %%
# Visualize diversity trend
fig, ax = plt.subplots(figsize=(14, 6))
diversity.plot(ax=ax, linewidth=2)
ax.set_title("Số lượng Tên chiếm 50% Tổng số Sinh (1880-2010)", 
             fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Số lượng Tên", fontsize=12)
ax.legend(["Nữ", "Nam"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH ĐA DẠNG HÓA ===")
print(f"Năm 1880: Nam cần {diversity.loc[1880, 'M']} tên, Nữ cần {diversity.loc[1880, 'F']} tên")
print(f"Năm 2010: Nam cần {diversity.loc[2010, 'M']} tên, Nữ cần {diversity.loc[2010, 'F']} tên")
print(f"\nTăng trưởng đa dạng Nam: {(diversity.loc[2010, 'M'] / diversity.loc[1880, 'M'] - 1) * 100:.1f}%")
print(f"Tăng trưởng đa dạng Nữ: {(diversity.loc[2010, 'F'] / diversity.loc[1880, 'F'] - 1) * 100:.1f}%")

# %% [markdown]
# ### 3.2. Phân tích Tên Trung tính Giới (Gender-Neutral Names)

# %%
# Tìm các tên xuất hiện ở cả hai giới
all_names = pd.Series(top1000["name"].unique())

# Tạo pivot table để so sánh
name_sex_births = top1000.groupby(['name', 'sex'])['births'].sum().unstack(fill_value=0)
name_sex_births['total'] = name_sex_births['F'] + name_sex_births['M']
name_sex_births['ratio_F'] = name_sex_births['F'] / name_sex_births['total']
name_sex_births['ratio_M'] = name_sex_births['M'] / name_sex_births['total']

# Tìm tên trung tính (tỷ lệ từ 0.3 đến 0.7)
neutral_names = name_sex_births[
    (name_sex_births['ratio_F'] >= 0.3) & 
    (name_sex_births['ratio_F'] <= 0.7) &
    (name_sex_births['total'] >= 10000)  # Lọc tên phổ biến
].sort_values('total', ascending=False)

print("=== TOP 20 TÊN TRUNG TÍNH GIỚI ===")
print(neutral_names.head(20)[['F', 'M', 'total', 'ratio_F', 'ratio_M']])

# %%
# Phân tích xu hướng tên trung tính theo thời gian
def calculate_neutrality(group):
    """Tính chỉ số trung tính: càng gần 1 càng trung tính"""
    f_births = group[group['sex'] == 'F']['births'].sum()
    m_births = group[group['sex'] == 'M']['births'].sum()
    total = f_births + m_births
    if total == 0:
        return 0
    ratio = min(f_births, m_births) / total
    return ratio * 2  # Scale 0-1

# Tính trung tính cho một số tên cụ thể
neutral_examples = ['Taylor', 'Jordan', 'Casey', 'Riley', 'Morgan']
neutral_examples = [n for n in neutral_examples if n in top1000['name'].values]

neutrality_trends = {}
for name in neutral_examples:
    name_data = top1000[top1000['name'] == name]
    yearly_neutrality = name_data.groupby('year').apply(calculate_neutrality)
    neutrality_trends[name] = yearly_neutrality

neutrality_df = pd.DataFrame(neutrality_trends)

if not neutrality_df.empty:
    fig, ax = plt.subplots(figsize=(14, 6))
    neutrality_df.plot(ax=ax, linewidth=2, marker='o', markersize=4)
    ax.set_title("Chỉ số Trung tính Giới của các Tên (1880-2010)", 
                 fontsize=16, fontweight='bold')
    ax.set_xlabel("Năm", fontsize=12)
    ax.set_ylabel("Chỉ số Trung tính (0-1)", fontsize=12)
    ax.set_ylim(0, 1.05)
    ax.legend(title="Tên", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Ngưỡng trung tính')
    plt.tight_layout()
    plt.show()
else:
    print("Không có dữ liệu về tên trung tính trong top1000")

# %% [markdown]
# ## 4. PHÂN TÍCH SO SÁNH

# %% [markdown]
# ### 4.1. So sánh Độ Tập trung giữa Bé Trai và Bé Gái

# %%
# Tính hệ số Gini để đo độ tập trung
def gini_coefficient(x):
    """Tính hệ số Gini: 0 = hoàn toàn đồng đều, 1 = hoàn toàn tập trung"""
    x = np.sort(x)
    n = len(x)
    cumsum = np.cumsum(x)
    return (2 * np.sum((n - np.arange(1, n+1) + 0.5) * x)) / (n * cumsum[-1]) - 1

# Tính Gini cho mỗi năm và giới tính
gini_by_year = names.groupby(['year', 'sex'])['births'].apply(gini_coefficient).unstack()

fig, ax = plt.subplots(figsize=(14, 6))
gini_by_year.plot(ax=ax, linewidth=2)
ax.set_title("Hệ số Gini của Phân phối Tên (1880-2010)", 
             fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Hệ số Gini", fontsize=12)
ax.legend(["Nữ", "Nam"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH HỆ SỐ GINI ===")
print("Hệ số Gini cao = độ tập trung cao = ít đa dạng")
print(f"\nNăm 1880: Nam = {gini_by_year.loc[1880, 'M']:.4f}, Nữ = {gini_by_year.loc[1880, 'F']:.4f}")
print(f"Năm 2010: Nam = {gini_by_year.loc[2010, 'M']:.4f}, Nữ = {gini_by_year.loc[2010, 'F']:.4f}")

# %% [markdown]
# ### 4.2. Phân tích Vòng đời của Tên (Name Lifespan)

# %%
# Tính số năm mỗi tên xuất hiện trong top 1000
name_years = top1000.groupby('name')['year'].agg(['min', 'max', 'count'])
name_years['lifespan'] = name_years['max'] - name_years['min'] + 1
name_years['appearance_rate'] = name_years['count'] / name_years['lifespan']

# Phân loại tên
name_years['category'] = pd.cut(name_years['lifespan'], 
                                 bins=[0, 20, 50, 100, 150],
                                 labels=['Short-lived (<20y)', 
                                        'Medium (20-50y)', 
                                        'Long (50-100y)', 
                                        'Very Long (>100y)'])

print("=== PHÂN LOẠI TÊN THEO VÒNG ĐỜI ===")
print(name_years['category'].value_counts().sort_index())

# %%
# Top 10 tên tồn tại lâu nhất
long_lived = name_years.nlargest(10, 'lifespan')
print("\n=== TOP 10 TÊN TỒN TẠI LÂU NHẤT ===")
print(long_lived[['min', 'max', 'lifespan', 'count', 'appearance_rate']])

# Top 10 tên ngắn hạn nhưng phổ biến
short_popular = name_years[(name_years['lifespan'] < 30) & 
                           (name_years['count'] >= 20)].nlargest(10, 'count')
print("\n=== TOP 10 TÊN NGẮN HẠN NHƯNG PHỔ BIẾN ===")
print(short_popular[['min', 'max', 'lifespan', 'count', 'appearance_rate']])

# %%
# Visualize lifespan distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(name_years['lifespan'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title("Phân phối Vòng đời của Tên", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Số năm tồn tại", fontsize=11)
axes[0].set_ylabel("Số lượng Tên", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Box plot by category
name_years.boxplot(column='count', by='category', ax=axes[1])
axes[1].set_title("Số lần xuất hiện theo Loại Vòng đời", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Loại Vòng đời", fontsize=11)
axes[1].set_ylabel("Số năm xuất hiện", fontsize=11)
plt.suptitle("")  # Remove default title

plt.tight_layout()
plt.show()

# %% [markdown]
# ## 5. PHÂN TÍCH CHỮ CÁI CUỐI (Last Letter Revolution)

# %%
# Tạo cột last_letter
def get_last_letter(x):
    return x[-1].lower()

names["last_letter"] = names["name"].map(get_last_letter)

# Pivot table theo last_letter
letter_table = names.pivot_table("births", index="last_letter",
                                  columns=["sex", "year"], aggfunc=sum)

# Chọn một số năm đại diện
representative_years = [1910, 1960, 2010]
subtable = letter_table.reindex(columns=representative_years, level="year")

# Normalize
letter_prop = subtable / subtable.sum()

print("=== TỶ LỆ TÊN THEO CHỮ CÁI CUỐI - MỘT SỐ NĂM ===")
print(letter_prop.head(10))

# %%
# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

letter_prop["M"].plot(kind="bar", ax=axes[0], width=0.8)
axes[0].set_title("Tỷ lệ Tên Bé Trai theo Chữ cái Cuối", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Chữ cái cuối", fontsize=11)
axes[0].set_ylabel("Tỷ lệ", fontsize=11)
axes[0].legend(title="Năm", fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

letter_prop["F"].plot(kind="bar", ax=axes[1], width=0.8)
axes[1].set_title("Tỷ lệ Tên Bé Gái theo Chữ cái Cuối", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Chữ cái cuối", fontsize=11)
axes[1].set_ylabel("Tỷ lệ", fontsize=11)
axes[1].legend(title="Năm", fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# %%
# Phân tích chi tiết chữ 'd', 'n', 'y' cho bé trai
letter_prop_all = letter_table / letter_table.sum()
dny_ts = letter_prop_all.loc[["d", "n", "y"], "M"].T

fig, ax = plt.subplots(figsize=(14, 6))
dny_ts.plot(ax=ax, linewidth=2)
ax.set_title("Xu hướng Chữ cái Cuối 'd', 'n', 'y' cho Bé Trai (1880-2010)", 
             fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Tỷ lệ", fontsize=12)
ax.legend(["d", "n", "y"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH CHỮ CÁI CUỐI ===")
print("Tên kết thúc bằng 'n' đã tăng mạnh từ những năm 1960")

# %% [markdown]
# ## 6. PHÂN TÍCH DỰ ĐOÁN

# %% [markdown]
# ### 6.1. Nhận diện Tên đang Tăng/Giảm

# %%
# Tính tốc độ tăng trưởng trong 20 năm gần nhất (1990-2010)
recent_data = names[names['year'] >= 1990]

# Tính tổng births cho mỗi tên trong 2 giai đoạn
early_period = recent_data[recent_data['year'].between(1990, 1999)]
late_period = recent_data[recent_data['year'].between(2000, 2010)]

early_births = early_period.groupby(['name', 'sex'])['births'].sum()
late_births = late_period.groupby(['name', 'sex'])['births'].sum()

# Tính tốc độ tăng trưởng
growth = pd.DataFrame({
    'early': early_births,
    'late': late_births
}).fillna(0)

growth['growth_rate'] = ((growth['late'] - growth['early']) / 
                         (growth['early'] + 1)) * 100  # +1 tránh chia 0
growth['total'] = growth['early'] + growth['late']

# Lọc tên phổ biến
popular_growth = growth[growth['total'] >= 5000].sort_values('growth_rate', ascending=False)

print("=== TOP 15 TÊN TĂNG TRƯỞNG NHANH NHẤT (1990-2010) ===")
print(popular_growth.head(15)[['early', 'late', 'growth_rate']])

print("\n=== TOP 15 TÊN GIẢM NHANH NHẤT (1990-2010) ===")
print(popular_growth.tail(15)[['early', 'late', 'growth_rate']])

# %% [markdown]
# ### 6.2. Phát hiện Tên "Vintage" (Quay trở lại)

# %%
# Tìm tên có pattern: phổ biến → giảm → tăng lại
def detect_vintage_pattern(name_data, threshold=0.3):
    """
    Phát hiện pattern vintage: popularity giảm rồi tăng lại
    """
    if len(name_data) < 30:
        return False
    
    # Tính moving average
    name_data = name_data.sort_values('year')
    name_data['ma_prop'] = name_data['prop'].rolling(window=5, center=True).mean()
    
    # Tìm min và max
    max_val = name_data['ma_prop'].max()
    min_idx = name_data['ma_prop'].idxmin()
    
    # Check nếu có sự phục hồi
    if min_idx < name_data.index[-1]:
        recovery = name_data.loc[min_idx:, 'ma_prop'].max()
        decline = name_data.loc[:min_idx, 'ma_prop'].min()
        if recovery > decline * (1 + threshold):
            return True
    return False

# Áp dụng cho tên phổ biến
vintage_candidates = []
for name in top1000['name'].unique()[:500]:  # Test với 500 tên đầu
    name_data = top1000[top1000['name'] == name]
    if detect_vintage_pattern(name_data):
        vintage_candidates.append(name)

print(f"=== PHÁT HIỆN {len(vintage_candidates)} TÊN CÓ PATTERN VINTAGE ===")
print("Ví dụ:")
print(vintage_candidates[:20])

# %%
# Visualize một số tên vintage điển hình
vintage_examples = vintage_candidates[:6] if len(vintage_candidates) >= 6 else vintage_candidates

if vintage_examples:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, name in enumerate(vintage_examples):
        name_data = top1000[top1000['name'] == name]
        yearly_prop = name_data.groupby('year')['prop'].sum()
        
        axes[idx].plot(yearly_prop.index, yearly_prop.values, linewidth=2)
        axes[idx].set_title(f"Tên: {name}", fontsize=12, fontweight='bold')
        axes[idx].set_xlabel("Năm", fontsize=10)
        axes[idx].set_ylabel("Tỷ lệ", fontsize=10)
        axes[idx].grid(True, alpha=0.3)
    
    plt.suptitle("Tên có Pattern 'Vintage' (Phổ biến → Giảm → Quay lại)", 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy tên vintage trong mẫu kiểm tra")

# %% [markdown]
# ### 6.3. Dự đoán Xu hướng bằng Linear Regression

# %%
# Chọn một số tên để dự đoán xu hướng
prediction_names = ['Emma', 'Olivia', 'Liam', 'Noah', 'Sophia']
prediction_names = [n for n in prediction_names if n in top1000['name'].values]

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

predictions_results = {}

for name in prediction_names:
    name_data = top1000[top1000['name'] == name]
    yearly_data = name_data.groupby('year')['births'].sum().reset_index()
    
    if len(yearly_data) < 10:
        continue
    
    # Prepare data
    X = yearly_data['year'].values.reshape(-1, 1)
    y = yearly_data['births'].values
    
    # Fit linear model
    model = LinearRegression()
    model.fit(X[-20:], y[-20:])  # Dùng 20 năm gần nhất
    
    # Predict next 10 years
    future_years = np.arange(2011, 2021).reshape(-1, 1)
    predictions = model.predict(future_years)
    
    predictions_results[name] = {
        'historical': yearly_data,
        'predictions': predictions,
        'trend': 'Tăng' if model.coef_[0] > 0 else 'Giảm',
        'slope': model.coef_[0]
    }

print("=== DỰ ĐOÁN XU HƯỚNG 2011-2020 ===")
for name, result in predictions_results.items():
    print(f"\nTên: {name}")
    print(f"  Xu hướng: {result['trend']}")
    print(f"  Tốc độ thay đổi: {result['slope']:.1f} births/năm")

# %%
# Visualize predictions
if predictions_results:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, (name, result) in enumerate(list(predictions_results.items())[:6]):
        historical = result['historical']
        
        # Plot historical data
        axes[idx].plot(historical['year'], historical['births'], 
                      linewidth=2, label='Dữ liệu thực tế', color='blue')
        
        # Plot predictions
        future_years = np.arange(2011, 2021)
        axes[idx].plot(future_years, result['predictions'], 
                      linewidth=2, linestyle='--', label='Dự đoán', color='red')
        
        axes[idx].set_title(f"Tên: {name} ({result['trend']})", 
                           fontsize=12, fontweight='bold')
        axes[idx].set_xlabel("Năm", fontsize=10)
        axes[idx].set_ylabel("Số lượng Sinh", fontsize=10)
        axes[idx].legend(fontsize=9)
        axes[idx].grid(True, alpha=0.3)
        axes[idx].axvline(x=2010, color='gray', linestyle=':', alpha=0.5)
    
    # Hide extra subplots
    for idx in range(len(predictions_results), 6):
        axes[idx].axis('off')
    
    plt.suptitle("Dự đoán Xu hướng Tên (2011-2020)", 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# %% [markdown]
# ### 6.4. Phân tích Chu kỳ Văn hóa (Cultural Cycles)

# %%
# Tính autocorrelation để phát hiện chu kỳ
from scipy.signal import find_peaks

def analyze_cycles(name_data, name):
    """Phân tích chu kỳ của một tên"""
    yearly_data = name_data.groupby('year')['prop'].sum().sort_index()
    
    if len(yearly_data) < 30:
        return None
    
    # Detrend data
    from scipy.signal import detrend
    detrended = detrend(yearly_data.values)
    
    # Find peaks
    peaks, properties = find_peaks(detrended, distance=15, prominence=0.0001)
    
    if len(peaks) > 1:
        # Tính khoảng cách trung bình giữa các peak
        cycle_length = np.mean(np.diff(yearly_data.index[peaks]))
        return {
            'name': name,
            'cycle_length': cycle_length,
            'num_peaks': len(peaks),
            'years': yearly_data.index.tolist(),
            'values': yearly_data.values.tolist(),
            'peak_years': yearly_data.index[peaks].tolist()
        }
    return None

# Phân tích chu kỳ cho các tên
cycle_analysis = []
for name in top1000['name'].unique()[:200]:  # Test với 200 tên
    name_data = top1000[top1000['name'] == name]
    result = analyze_cycles(name_data, name)
    if result and result['cycle_length'] > 0:
        cycle_analysis.append(result)

# Sắp xếp theo độ dài chu kỳ
cycle_df = pd.DataFrame(cycle_analysis)
if not cycle_df.empty:
    cycle_df = cycle_df.sort_values('cycle_length')
    
    print("=== PHÂN TÍCH CHU KỲ VĂN HÓA ===")
    print("\nTên có chu kỳ ngắn (thay đổi nhanh):")
    print(cycle_df.head(10)[['name', 'cycle_length', 'num_peaks']])
    
    print("\nTên có chu kỳ dài (thay đổi chậm):")
    print(cycle_df.tail(10)[['name', 'cycle_length', 'num_peaks']])

# %% [markdown]
# ## 7. TÓM TẮT VÀ KẾT LUẬN

# %%
print("=" * 80)
print("TÓM TẮT PHÂN TÍCH US BABY NAMES (1880-2010)")
print("=" * 80)

print("\n1. XU HƯỚNG TỔNG QUAN:")
print(f"   - Tổng số records: {len(names):,}")
print(f"   - Số tên unique: {names['name'].nunique():,}")
print(f"   - Tổng số sinh 1880: {total_births.loc[1880].sum():,.0f}")
print(f"   - Tổng số sinh 2010: {total_births.loc[2010].sum():,.0f}")

print("\n2. ĐỘ ĐA DẠNG:")
print(f"   - Năm 1880: Cần {diversity.loc[1880, 'M']} tên nam, {diversity.loc[1880, 'F']} tên nữ để đạt 50%")
print(f"   - Năm 2010: Cần {diversity.loc[2010, 'M']} tên nam, {diversity.loc[2010, 'F']} tên nữ để đạt 50%")
print(f"   - Tăng trưởng đa dạng: Nam {(diversity.loc[2010, 'M']/diversity.loc[1880, 'M']-1)*100:.0f}%, Nữ {(diversity.loc[2010, 'F']/diversity.loc[1880, 'F']-1)*100:.0f}%")

print("\n3. ĐỘ TẬP TRUNG:")
print(f"   - Top 1000 tên năm 1880: Nam {table.loc[1880, 'M']:.1%}, Nữ {table.loc[1880, 'F']:.1%}")
print(f"   - Top 1000 tên năm 2010: Nam {table.loc[2010, 'M']:.1%}, Nữ {table.loc[2010, 'F']:.1%}")
print("   → Độ tập trung giảm đáng kể, phản ánh xu hướng đa dạng hóa")

print("\n4. TÊN TRUNG TÍNH GIỚI:")
if not neutral_names.empty:
    print(f"   - Số tên trung tính (tỷ lệ 30-70%): {len(neutral_names)}")
    print(f"   - Tên trung tính phổ biến nhất: {neutral_names.index[0]}")

print("\n5. CHỮ CÁI CUỐI:")
print("   - Tên nam kết thúc bằng 'n' tăng mạnh từ 1960s")
print("   - Tên nữ kết thúc bằng 'a' vẫn chiếm ưu thế nhưng giảm dần")

print("\n6. VÒNG ĐỜI TÊN:")
print(f"   - Tên tồn tại trung bình: {name_years['lifespan'].mean():.1f} năm")
print(f"   - Tên tồn tại lâu nhất: {name_years['lifespan'].max()} năm")
print(f"   - Tên ngắn hạn (<20 năm): {(name_years['lifespan'] < 20).sum()}")

if vintage_candidates:
    print(f"\n7. TÊN VINTAGE:")
    print(f"   - Số tên có pattern quay trở lại: {len(vintage_candidates)}")
    print(f"   - Ví dụ: {', '.join(vintage_candidates[:5])}")

print("\n8. DỰ ĐOÁN XU HƯỚNG:")
if predictions_results:
    rising = sum(1 for r in predictions_results.values() if r['trend'] == 'Tăng')
    falling = sum(1 for r in predictions_results.values() if r['trend'] == 'Giảm')
    print(f"   - Trong {len(predictions_results)} tên được phân tích:")
    print(f"     • {rising} tên có xu hướng tăng")
    print(f"     • {falling} tên có xu hướng giảm")

print("\n" + "=" * 80)
print("KẾT LUẬN:")
print("=" * 80)
print("""
1. Đa dạng hóa tên ngày càng tăng, phản ánh sự thay đổi văn hóa và xã hội

2. Tên nữ luôn đa dạng hơn tên nam trong suốt 130 năm

3. Xu hướng tên trung tính giới tăng lên, đặc biệt từ những năm 1980

4. Có sự luân chuyển của tên theo chu kỳ văn hóa (vintage names)

5. Các yếu tố ảnh hưởng đến đặt tên:
   - Văn hóa đại chúng (phim ảnh, người nổi tiếng)
   - Di cư và đa văn hóa
   - Xu hướng cá nhân hóa và độc đáo
   - Yếu tố tôn giáo và truyền thống gia đình

6. Dự đoán tương lai:
   - Đa dạng hóa sẽ tiếp tục tăng
   - Tên trung tính giới sẽ phổ biến hơn
   - Các tên cổ điển có thể quay trở lại theo chu kỳ 30-50 năm
""")

# %% [markdown]
# ## 8. PHÂN TÍCH BỔ SUNG

# %% [markdown]
# ### 8.1. Độ dài Tên theo Thời gian

# %%
# Tính độ dài trung bình của tên
names['name_length'] = names['name'].str.len()

# Tính độ dài trung bình có trọng số theo số lượng sinh
def weighted_mean_length(group):
    return (group['name_length'] * group['births']).sum() / group['births'].sum()

avg_length = names.groupby(['year', 'sex']).apply(weighted_mean_length).unstack()

fig, ax = plt.subplots(figsize=(14, 6))
avg_length.plot(ax=ax, linewidth=2)
ax.set_title("Độ dài Trung bình của Tên (1880-2010)", 
             fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Số ký tự", fontsize=12)
ax.legend(["Nữ", "Nam"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH ĐỘ DÀI TÊN ===")
print(f"Năm 1880: Nam = {avg_length.loc[1880, 'M']:.2f}, Nữ = {avg_length.loc[1880, 'F']:.2f}")
print(f"Năm 2010: Nam = {avg_length.loc[2010, 'M']:.2f}, Nữ = {avg_length.loc[2010, 'F']:.2f}")

# %% [markdown]
# ### 8.2. Phân tích Tên theo Nguồn gốc (Approximate)

# %%
# Phân tích chữ cái đầu để ước lượng nguồn gốc
names['first_letter'] = names['name'].str[0].str.upper()

# Top chữ cái đầu theo thời gian
first_letter_prop = names.pivot_table('births', 
                                      index='first_letter', 
                                      columns=['year', 'sex'], 
                                      aggfunc='sum')

# Normalize
first_letter_prop = first_letter_prop.div(first_letter_prop.sum(axis=0), axis=1)

# Chọn một số năm
selected_years = [1900, 1950, 2000]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, year in enumerate(selected_years):
    year_data = first_letter_prop[(year, 'M')] + first_letter_prop[(year, 'F')]
    year_data = year_data.sort_values(ascending=False).head(10)
    
    year_data.plot(kind='bar', ax=axes[idx], color='steelblue', edgecolor='black')
    axes[idx].set_title(f"Top 10 Chữ cái Đầu - Năm {year}", 
                       fontsize=12, fontweight='bold')
    axes[idx].set_xlabel("Chữ cái", fontsize=10)
    axes[idx].set_ylabel("Tỷ lệ", fontsize=10)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# %% [markdown]
# ### 8.3. Tính Entropy để Đo Đa dạng

# %%
from scipy.stats import entropy

def calculate_entropy(group):
    """Tính Shannon entropy"""
    props = group / group.sum()
    return entropy(props)

# Tính entropy cho mỗi năm
entropy_by_year = names.groupby(['year', 'sex'])['births'].apply(calculate_entropy).unstack()

fig, ax = plt.subplots(figsize=(14, 6))
entropy_by_year.plot(ax=ax, linewidth=2)
ax.set_title("Shannon Entropy của Phân phối Tên (1880-2010)", 
             fontsize=16, fontweight='bold')
ax.set_xlabel("Năm", fontsize=12)
ax.set_ylabel("Entropy (bits)", fontsize=12)
ax.legend(["Nữ", "Nam"], fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH ENTROPY ===")
print("Entropy cao = đa dạng cao")
print(f"Năm 1880: Nam = {entropy_by_year.loc[1880, 'M']:.2f}, Nữ = {entropy_by_year.loc[1880, 'F']:.2f}")
print(f"Năm 2010: Nam = {entropy_by_year.loc[2010, 'M']:.2f}, Nữ = {entropy_by_year.loc[2010, 'F']:.2f}")

# %% [markdown]
# ### 8.4. Phân tích Tên "Unisex" Evolution

# %%
# Theo dõi sự thay đổi giới tính của tên theo thời gian
def track_gender_shift(name, data):
    """Theo dõi tỷ lệ giới tính của một tên theo thời gian"""
    name_data = data[data['name'] == name]
    pivot = name_data.pivot_table('births', index='year', columns='sex', 
                                   aggfunc='sum', fill_value=0)
    pivot['total'] = pivot.sum(axis=1)
    if 'F' in pivot.columns:
        pivot['pct_female'] = pivot['F'] / pivot['total']
    else:
        pivot['pct_female'] = 0
    return pivot

# Chọn một số tên để phân tích
example_names = ['Leslie', 'Jordan', 'Taylor', 'Casey', 'Riley', 'Morgan']
example_names = [n for n in example_names if n in top1000['name'].values]

if example_names:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, name in enumerate(example_names[:6]):
        gender_trend = track_gender_shift(name, top1000)
        
        axes[idx].plot(gender_trend.index, gender_trend['pct_female'] * 100, 
                      linewidth=2, color='purple')
        axes[idx].axhline(y=50, color='red', linestyle='--', alpha=0.5, 
                         label='50% (Trung tính)')
        axes[idx].fill_between(gender_trend.index, 0, 
                              gender_trend['pct_female'] * 100, 
                              alpha=0.3, color='pink', label='% Nữ')
        axes[idx].set_title(f"Tên: {name}", fontsize=12, fontweight='bold')
        axes[idx].set_xlabel("Năm", fontsize=10)
        axes[idx].set_ylabel("% Nữ", fontsize=10)
        axes[idx].set_ylim(0, 100)
        axes[idx].legend(fontsize=8)
        axes[idx].grid(True, alpha=0.3)
    
    # Hide extra subplots
    for idx in range(len(example_names), 6):
        axes[idx].axis('off')
    
    plt.suptitle("Sự Chuyển đổi Giới tính của Tên Unisex", 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# %% [markdown]
# ### 8.5. Correlation Analysis giữa các Metrics

# %%
# Tạo summary metrics cho mỗi năm
yearly_metrics = pd.DataFrame({
    'year': range(1880, 2011),
    'total_births_M': [total_births.loc[y, 'M'] for y in range(1880, 2011)],
    'total_births_F': [total_births.loc[y, 'F'] for y in range(1880, 2011)],
    'diversity_M': [diversity.loc[y, 'M'] for y in range(1880, 2011)],
    'diversity_F': [diversity.loc[y, 'F'] for y in range(1880, 2011)],
    'top1000_prop_M': [table.loc[y, 'M'] for y in range(1880, 2011)],
    'top1000_prop_F': [table.loc[y, 'F'] for y in range(1880, 2011)],
    'gini_M': [gini_by_year.loc[y, 'M'] for y in range(1880, 2011)],
    'gini_F': [gini_by_year.loc[y, 'F'] for y in range(1880, 2011)],
    'entropy_M': [entropy_by_year.loc[y, 'M'] for y in range(1880, 2011)],
    'entropy_F': [entropy_by_year.loc[y, 'F'] for y in range(1880, 2011)],
    'avg_length_M': [avg_length.loc[y, 'M'] for y in range(1880, 2011)],
    'avg_length_F': [avg_length.loc[y, 'F'] for y in range(1880, 2011)]
})

# Tính correlation matrix
corr_matrix = yearly_metrics.corr()

# Visualize
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Ma trận Tương quan giữa các Chỉ số", 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n=== PHÂN TÍCH TƯƠNG QUAN ===")
print("\nTương quan mạnh nhất:")
# Lấy upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_values = corr_matrix.where(mask).stack().sort_values(ascending=False)
print(corr_values.head(10))

# %% [markdown]
# ## 9. LƯU KẾT QUẢ

# %%
# Export các kết quả quan trọng
results_dir = "baby_names_results"
import os
os.makedirs(results_dir, exist_ok=True)

# 1. Top 1000 names dataset
top1000.to_csv(f"{results_dir}/top1000_names.csv", index=False)
print(f"✓ Đã lưu: {results_dir}/top1000_names.csv")

# 2. Yearly metrics
yearly_metrics.to_csv(f"{results_dir}/yearly_metrics.csv", index=False)
print(f"✓ Đã lưu: {results_dir}/yearly_metrics.csv")

# 3. Name lifespan analysis
name_years.to_csv(f"{results_dir}/name_lifespan.csv")
print(f"✓ Đã lưu: {results_dir}/name_lifespan.csv")

# 4. Gender-neutral names
if not neutral_names.empty:
    neutral_names.to_csv(f"{results_dir}/neutral_names.csv")
    print(f"✓ Đã lưu: {results_dir}/neutral_names.csv")

# 5. Growth analysis
popular_growth.to_csv(f"{results_dir}/name_growth_1990_2010.csv")
print(f"✓ Đã lưu: {results_dir}/name_growth_1990_2010.csv")

print(f"\n✓ Tất cả kết quả đã được lưu vào thư mục: {results_dir}/")

# %%
print("\n" + "="*80)
print("HOÀN THÀNH PHÂN TÍCH!")
print("="*80)
print("""
Notebook này đã phân tích toàn diện dữ liệu US Baby Names với:
✓ Phân tích mô tả: Xu hướng tổng quan, top names, độ tập trung
✓ Phân tích khám phá: Đa dạng hóa, tên trung tính giới
✓ Phân tích so sánh: Gini coefficient, vòng đời tên
✓ Phân tích dự đoán: Xu hướng tăng/giảm, vintage names, chu kỳ văn hóa
✓ Phân tích bổ sung: Độ dài tên, entropy, tương quan

Các file kết quả đã được export để sử dụng tiếp.
""")